1. Ознакомьтесь с датасетом образцов эмоциональной речи

    **Toronto emotional speech set (TESS)**:

    https://dataverse.scholarsportal.info/dataset.xhtml?persistentId=doi:10.5683/SP2/E8H2MF

    Ссылка для загрузки данных: https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip

2. Разберите датасет;
3. Подготовьте и разделите данные на обучающие и тестовые;
4. Разработайте классификатор, показывающий на тесте точность распознавания эмоции не менее 98%;
5. Ознакомьтесь с другим датасетом похожего содержания

    **Surrey Audio-Visual Expressed Emotion (SAVEE)**:

    https://www.kaggle.com/ejlok1/surrey-audiovisual-expressed-emotion-savee

    Ссылка для загрузки данных: https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip

6. Прогоните обученный классификатор на файлах из датасета **SAVEE** по вашему выбору;
7. Сделайте выводы.

## 1. Импорт библиотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import librosa
import os
import zipfile
import gdown
import warnings
warnings.filterwarnings('ignore')

SR = 22050
DURATION = 3

## 2. Загрузка и распаковка TESS

In [39]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip', 'dataverse_files.zip', quiet=False)

with zipfile.ZipFile('dataverse_files.zip', 'r') as zip_ref:
    zip_ref.extractall('TESS_data')

for root, dirs, files in os.walk('TESS_data'):
    if any(f.endswith('.wav') for f in files):
        tess_path = root
        break

print(f"Путь: {tess_path}")

Downloading...
From: https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip
To: /content/dataverse_files.zip
100%|██████████| 224M/224M [00:10<00:00, 20.9MB/s]


Путь: TESS_data


## 3. Функция загрузки TESS с аугментацией (шум, сдвиг)


In [40]:
def load_tess_augmented(path):
    X, y = [], []
    for file in os.listdir(path):
        if not file.endswith('.wav'):
            continue
        emotion = file.split('_')[-1].replace('.wav', '')
        if emotion == 'ps':
            emotion = 'surprise'
        file_path = os.path.join(path, file)
        try:
            audio, sr = librosa.load(file_path, sr=SR, duration=DURATION)
            if len(audio) < SR * DURATION:
                audio = np.pad(audio, (0, SR*DURATION - len(audio)))

            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
            feat = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
            X.append(feat)
            y.append(emotion)

            noise = np.random.normal(0, 0.005, len(audio))
            audio_noise = audio + noise
            mfcc = librosa.feature.mfcc(y=audio_noise, sr=sr, n_mfcc=20)
            feat = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
            X.append(feat)
            y.append(emotion)

            shift = int(np.random.uniform(0.1, 0.3) * sr)
            audio_shift = np.roll(audio, shift)
            mfcc = librosa.feature.mfcc(y=audio_shift, sr=sr, n_mfcc=20)
            feat = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
            X.append(feat)
            y.append(emotion)
        except:
            pass
    return np.array(X), np.array(y)

X_tess, y_tess = load_tess_augmented(tess_path)
print(f"TESS: {X_tess.shape[0]} файлов, признаки: {X_tess.shape[1]}")
print(f"Эмоции: {np.unique(y_tess)}")

TESS: 8400 файлов, признаки: 40
Эмоции: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']


## 4. Разделение на train/test (80/20)

In [41]:
le = LabelEncoder()
y_tess_enc = le.fit_transform(y_tess)
y_tess_cat = to_categorical(y_tess_enc)

X_train, X_test, y_train, y_test = train_test_split(X_tess, y_tess_cat, test_size=0.2, stratify=y_tess_enc, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (6720, 40), Test: (1680, 40)


## 5. Создание модели

In [42]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(40,)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           119 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,607 (21.90 KB)

 Trainable params: 5,479 (21.40 KB)

 Non-trainable params: 128 (512.00 B)

## 6. Обучение модели

In [43]:
early_stop = EarlyStopping(monitor='val_accuracy', patience=30, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)

history = model.fit(X_train, y_train, validation_split=0.1, batch_size=32, epochs=150, callbacks=[early_stop, reduce_lr], verbose=1)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Точность на TESS: {test_acc:.4f} ({test_acc*100:.2f}%)")

Epoch 1/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.2688 - loss: 1.9867 - val_accuracy: 0.7396 - val_loss: 1.3467 - learning_rate: 0.0010
Epoch 2/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4785 - loss: 1.3884 - val_accuracy: 0.8646 - val_loss: 0.7375 - learning_rate: 0.0010
Epoch 3/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6116 - loss: 1.0374 - val_accuracy: 0.9167 - val_loss: 0.3716 - learning_rate: 0.0010
Epoch 4/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7047 - loss: 0.8076 - val_accuracy: 0.9673 - val_loss: 0.2337 - learning_rate: 0.0010
Epoch 5/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7525 - loss: 0.6699 - val_accuracy: 0.9732 - val_loss: 0.1594 - learning_rate: 0.0010
Epoch 6/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7865 - loss: 0.5988 - val_accuracy: 0.9792 - val_loss: 0.1232 - learning_rate: 0.0010
Epoch 7/150
189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8080 - loss: 0

## 7. Загрузка и распаковка SAVEE

In [44]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip', 'archive.zip', quiet=False)

with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('SAVEE_data')

Downloading...
From: https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip
To: /content/archive.zip
100%|██████████| 113M/113M [00:09<00:00, 11.5MB/s]


## 8. Функция загрузки SAVEE с маппингом эмоций

In [45]:
def load_savee():
    X, y = [], []
    emotion_map = {'a': 'angry', 'd': 'disgust', 'f': 'fear', 'h': 'happy', 'n': 'neutral', 'sa': 'sad', 'su': 'surprise'}

    for root, dirs, files in os.walk('SAVEE_data'):
        for file in files:
            if not file.endswith('.wav'):
                continue
            parts = file.split('_')
            if len(parts) < 2:
                continue
            code = parts[1][:2] if parts[1].startswith('su') else parts[1][0]
            if code not in emotion_map:
                continue
            emotion = emotion_map[code]
            file_path = os.path.join(root, file)
            try:
                audio, sr = librosa.load(file_path, sr=SR, duration=DURATION)
                if len(audio) < SR * DURATION:
                    audio = np.pad(audio, (0, SR*DURATION - len(audio)))
                mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
                feat = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
                X.append(feat)
                y.append(emotion)
            except:
                pass
    return np.array(X), np.array(y)

X_savee, y_savee = load_savee()
print(f"SAVEE: {X_savee.shape[0]} файлов")
print(f"Эмоции: {np.unique(y_savee)}")

SAVEE: 420 файлов
Эмоции: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'surprise']


## 9. Предсказание на SAVEE

In [46]:
X_savee_scaled = scaler.transform(X_savee)

try:
    y_savee_enc = le.transform(y_savee)
    y_pred = np.argmax(model.predict(X_savee_scaled), axis=1)
    acc = accuracy_score(y_savee_enc, y_pred)
    print(f"Точность на SAVEE: {acc:.4f} ({acc*100:.2f}%)")
except Exception as e:
    print(f"Ошибка: {e}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
Точность на SAVEE: 0.0952 (9.52%)


## 10. Вывод

- **TESS: 98.87%** — требование ≥98% выполнено. Модель успешно распознаёт 7 эмоций (angry, disgust, fear, happy, neutral, sad, surprise) на тестовой выборке датасета TESS.

- **SAVEE: 9.52%** — при переносе на другой датасет без дообучения точность снизилась. Причины:
  - TESS записан двумя женщинами (60+ лет), SAVEE — четырьмя мужчинами (27-31 год)
  - Разная манера произнесения эмоций (утрированная vs сдержанная)
  - Разные условия записи (микрофон, помещение, уровень шума)